# **Build a LLM From Scratch - Sebastian Raschka**

#### Verify PyTorch and CUDA

In [1]:
import torch
import multiprocessing
print(torch.__version__)
print(torch.cuda.is_available())
multiprocessing.cpu_count()

2.11.0+cu128
True


12

#### Configuration

In [5]:
import os

CORES = 14
VOCAB_SIZE = 5000
SAMPLE_BYTES = 30 * 1024 * 1024
TOKENIZER_BYTES = 500 * 1024 * 1024
PRETRAINING_GB = 10 * 1024 * 1024 * 1024
SAMPLE_FILE = "academic_sample.txt"
TOKENIZER_FILE = "academic_tokenizing.txt"
PRETRAINING_FILE = "academic_pretraining.txt"
DATA = [SAMPLE_FILE, SAMPLE_BYTES]

#### Load and store data

In [ ]:
os.environ["HF_TOKEN"] = ""

from datasets import load_dataset
print("Connecting to Hugging Face dataset stream...")
dataset = load_dataset(
    "allenai/peS2o", split="train", streaming=True, trust_remote_code=True
)

bytes_written = 0
line_count = 0
print(f"Writing to file {DATA[0]}...")

with open(DATA[0], "w", encoding="utf-8") as f:
    for r in dataset:
        raw = r.get("text", "").strip()

        if raw:
            text = " ".join(raw.splitlines())
            to_write = text + "\n"
            f.write(to_write)

            bytes_written += len(to_write.encode("utf-8"))
            line_count += 1

            if line_count % 15000 == 0:
                mb_written = bytes_written / (1024 * 1024)
                print(f"Written: {mb_written:.2f} MB ({line_count} documents)")

            if bytes_written >= DATA[1]:
                print("Max bytes reached. Disconnecting stream")
                break

print(f"Done.\nFile saved as {DATA[0]}, {bytes_written / (1024 * 1024):.2f} MB")

Connecting to Hugging Face dataset stream...
Writing to file academic_sample.txt...
Written: 19.00 MB (15000 documents)
Max bytes reached. Disconnecting stream
Done.
File saved as academic_sample.txt, 30.00 MB


In [7]:
import pathlib
if os.path.isfile(DATA[0]):
    with open(DATA[0], "r", encoding="utf-8") as f:
        raw = f.readlines()
    TOKENIZER_DATA = [l.rstrip() for l in raw]
    print("Total no. characters:", len(raw))
    print(TOKENIZER_DATA[0])
else:
    print(f"File {pathlib.Path(DATA[0]).resolve()} not found")

Total no. characters: 23689
[Short-term effectiveness of anterior cruciate ligament reconstruction with LARs artificial ligament].  OBJECTIVE To investigate the surgical technique and short-term effectiveness of anterior cruciate ligament (ACL) reconstruction with LARS artificial ligament.   METHODS Between November 2008 and April 2010, eighty patients with ACL injury were treated with LARS artificial ligament under arthroscope and successfully followed up. There were 51 males and 29 females, aged from 17 to 43 years with an average of 29.2 years. The injuries were caused by sport in 63 cases, traffic accident in 14 cases, and bruise in 3 cases. There were 43 left knees and 37 right knees. The disease duration ranged from 10 days to 11 months. The anterior drawer test, Lachman test, and pivot shift test for all cases were rated as positive. The preoperative Lysholm score was 55.4 +/- 5.7, Irgang score was 48.3 +/- 6.2, and Larson score was 54.8 +/- 7.4; and the International Knee Docum

## Tokenizer

In [ ]:
from collections import defaultdict
from multiprocessing import Pool, cpu_count
from functools import lru_cache
import heapq
import ast
import re

pattern = re.compile(
    r'<\|endoftext\|>|<\|unk\|>|---|--|...|..|\s*[A-Za-z0-9]+|[()[],._?!"\'-]'
)


def pre_tokenize(line):
    words = pattern.findall(line)
    return [w.replace(" ", "G̃") for w in words]


def _worker(lines):
    local = defaultdict(int)
    for line in lines:
        for w in pre_tokenize(line):
            local[w] += 1
    return local


def compute_word_freqs(data):
    print("Starting pre-tokenization...")
    chunk_size = (len(data) + CORES - 1) // CORES
    chunks_data = [data[i : i + chunk_size] for i in range(0, len(data), chunk_size)]

    with Pool(cpu_count()) as pool:
        results = pool.map(_worker, chunks_data)

    print("Chunks done")

    word_freqs = defaultdict(int)
    for r in results:
        for k, v in r.items():
            word_freqs[k] += v

    print("Pre-tokenization done")

    return word_freqs


def base_vocab(word_freqs):
    alphabet = []
    for w in word_freqs.keys():
        for l in w:
            if l not in alphabet:
                alphabet.append(l)
    alphabet.sort()
    print("Base vocabulary initialized")
    return ["<|endoftext|>", "<|unk|>"] + alphabet.copy()


def init_structures(word_freqs):
    splits = {w: list(w) for w in word_freqs}

    pair_freqs = defaultdict(int)
    pair_positions = defaultdict(set)

    for w, f in word_freqs.items():
        tokens = splits[w]

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_freqs[pair] += f
            pair_positions[pair].add((w, i))

    heap = [(-f, pair) for pair, f in pair_freqs.items()]
    heapq.heapify(heap)

    return splits, pair_freqs, pair_positions, heap


def merge(pair, splits, pair_freqs, pair_positions, heap, word_freqs):
    newi = "".join([*pair])
    affected = list(pair_positions[pair])
    pair_positions.pop(pair, None)

    for w, i in affected:
        tokens = splits[w]
        if i >= len(tokens) - 1 or tokens[i] != pair[0] or tokens[i + 1] != pair[1]:
            continue

        freq = word_freqs[w]

        if i > 0:
            l = (tokens[i - 1], tokens[i])
            pair_freqs[l] -= freq
            pair_positions[l].discard((w, i - 1))

        if i + 2 < len(tokens):
            r = (tokens[i + 1], tokens[i + 2])
            pair_freqs[r] -= freq
            pair_positions[r].discard((w, i + 1))

        tokens[i : i + 2] = [newi]

        if i > 0:
            lnew = (tokens[i - 1], newi)
            pair_freqs[lnew] += freq
            pair_positions[lnew].add((w, i - 1))
            heapq.heappush(heap, (-pair_freqs[lnew], lnew))

        if i < len(tokens) - 1:
            rnew = (newi, tokens[i + 1])
            pair_freqs[rnew] += freq
            pair_positions[rnew].add((w, i))
            heapq.heappush(heap, (-pair_freqs[rnew], rnew))


def create_vocab(data, vocab_size):
    word_freqs = compute_word_freqs(data)
    vocab = base_vocab(word_freqs)

    splits, pair_freqs, pair_positions, heap = init_structures(word_freqs)
    merges = {}

    while len(vocab) < vocab_size and heap:
        freq, pair = heapq.heappop(heap)
        freq = -freq

        if pair_freqs[pair] != freq or freq == 0:
            continue

        merge(pair, splits, pair_freqs, pair_positions, heap, word_freqs)

        newi = "".join([*pair])
        merges[pair] = newi
        vocab.append(newi)

        if len(vocab) % 2000 == 0:
            print("Vocab size:", len(vocab))

    token_to_id = {t: i for i, t in enumerate(vocab)}
    id_to_token = {i: t for t, i in token_to_id.items()}

    return token_to_id, id_to_token, vocab, merges


class BPEtokenizer:
    def __init__(self, data=TOKENIZER_DATA, vocab_size=VOCAB_SIZE):
        run = int(input("New (1) or backed up vocabulary (2)? "))
        if run == 1:
            config = create_vocab(data, vocab_size)
        elif run == 2:
            with open("tokenizer_settings.txt", "r", encoding="utf-8") as f:
                config = ast.literal_eval(f.read())
        else:
            print("INVALID INPUT")
            exit()

        self.token_to_id, self.id_to_token, self.vocab, self.merges = config
        self.merge_ranks = {pair: i for i, pair in enumerate(self.merges)}

    def encode_word(self, word):
        if word in {"<|endoftext|>", "<|unk|>"}:
            return [self.token_to_id[word]]

        tokens = list(word)
        if len(tokens) <= 1:
            return [self.token_to_id.get(word, self.unk_id)]

        while True:
            best_rank = None
            best_idx = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.merge_ranks.get(pair)
                if rank is not None:
                    if best_rank is None or rank < best_rank:
                        best_rank = rank
                        best_idx = i

            if best_rank is None:
                break
            tokens[best_idx : best_idx + 2] = [tokens[best_idx] + tokens[best_idx + 1]]

        return tokens

    @lru_cache(maxsize=200_000)
    def encode_word_cached(self, word):
        return tuple(self.encode_word(word))

    def tokenize(self, text):
        words = pre_tokenize(text)
        tokens = []
        for w in words:
            tokens.extend(self.encode_word_cached(w))

        return list(tokens)

    def encode(self, text):
        tokens = self.tokenize(text)
        return [self.token_to_id.get(t, 1) for t in tokens]

    def decode(self, ids):
        text = "".join([self.id_to_token[x] for x in ids])
        return text.replace("G̃", " ")


if __name__ == "__main__":
    print("Cores available:", cpu_count())
    tokenizer = BPEtokenizer()

Cores available: 12
['The', 'G̃', '(', '[', 'dom', 'ina', 'n', 't', ']', ')G̃s', 'equ', 'enc', 'eG̃t', 'ran', 's', 'd', 'u', 'c', 't', 'i', 'onG̃', 'm', 'o', 'd', 'els', 'G̃', 'ar', 'eG̃b', 'a', 's', 'e', 'dG̃o', 'n', 'G̃', 'c', 'omp', 'lex', 'G̃', 'r', 'e', 'cur', 'ren', 'tG̃o', 'r', 'G̃', 'c', 'onv', 'olu', 't', 'i', 'o', 'nal', 'G̃', 'n', 'e', 'ura', 'l', 'G̃', 'n', 'etw', 'ork', 'sG̃t', 'h', 'a', 't', 'G̃', 'in', 'c', 'l', 'u', 'deG̃', 'anG̃', 'enc', 'o', 'd', 'e', 'r', 'G̃', 'a', 'ndG̃', 'aG̃d', 'eco', 'der', '.G̃T', 'heG̃', 'b', 'e', 's', 'tG̃p', 'erf', 'orm', 'ing', 'G̃', 'm', 'o', 'del', 'sG̃a', 'l', 's', 'o', 'G̃', 'c', 'o', 'n', 'n', 'e', 'ctG̃', 'the', 'G̃', 'en', 'c', 'o', 'd', 'erG̃', 'and', 'G̃', 'd', 'e', 'c', 'o', 'd', 'erG̃', 'thr', 'o', 'u', 'g', 'h', 'G̃', 'a', 'n', 'G̃', 'a', 't', 't', 'e', 'n', 't', 'i', 'onG̃', 'mec', 'han', 'ism', '.G̃W', 'eG̃p', 'r', 'o', 'p', 'o', 's', 'e', 'G̃', 'aG̃', 'new', 'G̃', 's', 'i', 'mpl', 'eG̃n', 'etw', 'ork', 'G̃', 'ar', 'c', 'h', '

In [13]:
print(
    tokenizer.tokenize(
        "The ([dominant]) sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train."
    )
)
ids = tokenizer.encode(
    "The ([dominant]) sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train."
)
print(ids)
print(len(ids))
print(tokenizer.decode(ids))

['The', 'G̃', '(', '[', 'dom', 'ina', 'n', 't', ']', ')G̃s', 'equ', 'enc', 'eG̃t', 'ran', 's', 'd', 'u', 'c', 't', 'i', 'onG̃', 'm', 'o', 'd', 'els', 'G̃', 'ar', 'eG̃b', 'a', 's', 'e', 'dG̃o', 'n', 'G̃', 'c', 'omp', 'lex', 'G̃', 'r', 'e', 'cur', 'ren', 'tG̃o', 'r', 'G̃', 'c', 'onv', 'olu', 't', 'i', 'o', 'nal', 'G̃', 'n', 'e', 'ura', 'l', 'G̃', 'n', 'etw', 'ork', 'sG̃t', 'h', 'a', 't', 'G̃', 'in', 'c', 'l', 'u', 'deG̃', 'anG̃', 'enc', 'o', 'd', 'e', 'r', 'G̃', 'a', 'ndG̃', 'aG̃d', 'eco', 'der', '.G̃T', 'heG̃', 'b', 'e', 's', 'tG̃p', 'erf', 'orm', 'ing', 'G̃', 'm', 'o', 'del', 'sG̃a', 'l', 's', 'o', 'G̃', 'c', 'o', 'n', 'n', 'e', 'ctG̃', 'the', 'G̃', 'en', 'c', 'o', 'd', 'erG̃', 'and', 'G̃', 'd', 'e', 'c', 'o', 'd', 'erG̃', 'thr', 'o', 'u', 'g', 'h', 'G̃', 'a', 'n', 'G̃', 'a', 't', 't', 'e', 'n', 't', 'i', 'onG̃', 'mec', 'han', 'ism', '.G̃W', 'eG̃p', 'r', 'o', 'p', 'o', 's', 'e', 'G̃', 'aG̃', 'new', 'G̃', 's', 'i', 'mpl', 'eG̃n', 'etw', 'ork', 'G̃', 'ar', 'c', 'h', 'i', 'tec', 'tur', 'e

##### Save tokenizer configuration

In [11]:
settings = [
    tokenizer.token_to_id,
    tokenizer.id_to_token,
    tokenizer.vocab,
    tokenizer.merges,
]
with open("tokenizer_settings.txt", "w", encoding="utf-8") as f:
    f.write(f"{settings}")

## Dataset and dataloader

In [9]:
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        tokens = torch.tensor(tokenizer.encode(txt), dtype=torch.long)
        windows = tokens.unfold(0, max_length + 1, stride)
        print("Data encoded")

        self.input_ids = windows[:, :-1]
        self.target_ids = windows[:, 1:]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    print("Dataset created")
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    print("Dataloader created")
    return dataloader


if __name__ == "__main__":
    print("Tokenizer initialized")
    dataloader = create_dataloader(
        "<|endoftext|>".join(TOKENIZER_DATA[:5]),
        batch_size=8,
        max_length=4,
        stride=4,
        shuffle=False,
        num_workers=12,
    )
    data_iter = iter(dataloader)
    inputs, targets = next(data_iter)
    print("Inputs:\n", inputs)
    print("\nTargets:\n", targets)

Tokenizer initialized
Data encoded
Dataset created
Dataloader created
Inputs:
 tensor([[  65, 1961,  895,   19],
        [  90,   75,   88,   83],
        [ 686, 1027,  766,  783],
        [  84,   75,   89,  761],
        [  76,  686,   71,   84],
        [  90,   75,   88,   79],
        [  85,   88,  686,   73],
        [1203,   79,   71,   90]])

Targets:
 tensor([[1961,  895,   19,   90],
        [  75,   88,   83,  686],
        [1027,  766,  783,   84],
        [  75,   89,  761,   76],
        [ 686,   71,   84,   90],
        [  75,   88,   79,   85],
        [  88,  686,   73, 1203],
        [  79,   71,   90,  977]])
